# 1019 — Layer-wise semantic-distance concept probing

**RQ:** How does a cue's semantic distance affect X-vs-Y internal discriminability under suppression?

Extension of [1015 v3](../1015/white_bear_and_zhou_pilot_v3.ipynb), with eight concept pairs, four graded cue categories and independent semantic-distance estimates. Probe scores are not absolute activation, conscious thoughts, nor evidence of causal reactivation. The target X appears in the instruction. Colab T4 GPU recommended.

Files are saved locally in `GraduationThesis/1019/results/<UTC-run-id>/`. GitHub upload needs the Colab secret `GITHUB_RESULTS_TOKEN`, with repository Contents write permission. A local ZIP is always made, even without the token.


## 0 — Colab setup
Use Runtime → Change runtime type → T4 GPU. Do not reinstall PyTorch; the following removes only optional incompatible audio/vision wheels. Run all cells top-to-bottom.

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install "transformers>=4.54,<5" "scikit-learn>=1.3" "pandas>=2.1" "matplotlib>=3.8" "requests>=2.31" "tqdm>=4.65"

In [ ]:
import os, json, re, base64, zipfile, random, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from tqdm.auto import tqdm

SEED=42
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
EMBED_ID="sentence-transformers/all-MiniLM-L6-v2"
LAYERS=[4,8,12,16,20,24]   # hidden_states[0]=embedding
LEVELS=["near","medium","far","unrelated"] # hypothetical group labels
CONDITIONS=["neutral","affirm","suppression","replacement"]
MIN_DESCRIPTION_AUC=0.75
RUN_GENERATION=False # set True for optional supplemental target-word mention analysis
GEN_REPEATS=1
UPLOAD_TO_GITHUB=True
REPO="ryouy/GraduationThesis"
BRANCH="main"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
if DEVICE=="cpu":print("WARNING: No GPU; switch Colab runtime to T4 for practical speed.")
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BASE=Path("/content/GraduationThesis") if Path("/content").is_dir() else Path.cwd()/"GraduationThesis"
OUT=BASE/"1019"/"results"/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
def save_csv(frame,name):
    frame.to_csv(OUT/name,index=False)
    print("Saved",name,len(frame),"rows")
def save_plot(name):
    plt.tight_layout();plt.savefig(OUT/name,dpi=180,bbox_inches="tight");plt.show();plt.close()
print("Device:",DEVICE,"Results:",OUT)

## 1 — Cue bank (fixed before modeling)

Eight concept pairs inherited from 1015 v3. For each X, four proposed distance bins contain **two distinct English sentences**. All cue sentences are checked not to contain the literal X or Y. The bins mix descriptive and associative relations and must not be treated as validated semantic distances.

In [ ]:
# Compact data: x, y, three x-descriptions, three y-descriptions, and 4 × 2 cue sentences.
ITEMS=[
("banana","apple",
 ["a long sweet yellow produce item","a soft fruit that is peeled before eating","a curved tropical snack fruit"],
 ["a crisp round orchard fruit","a red or green fruit with a core","a firm tree-grown fruit used in pies"],
 [
 ["The shopper peeled a long yellow fruit for a snack.","A curved yellow fruit lay beside the cereal bowl."],
 ["A monkey searched the canopy for something sweet to eat.","A smoothie recipe called for ripe tropical fruit."],
 ["The rainforest canopy was full of small birds.","A stall displayed imported produce in the morning."],
 ["A clerk stamped documents at a railway office.","Workers repaired a cracked pavement outside the library."]]),
("elephant","giraffe",
 ["a giant gray mammal with a long flexible nose","a tusked herbivore with exceptionally large ears","a heavy savanna animal famous for its trunk"],
 ["a tall spotted animal with an unusually long neck","a long-legged hoofed mammal browsing high branches","a graceful African animal with patterned patches"],
 [
 ["A giant gray mammal swung its trunk by a waterhole.","Huge ears flapped as a tusked animal crossed the plain."],
 ["A safari guide discussed the world's largest land mammals.","A herd of enormous herbivores crossed the savanna."],
 ["Tourists visited a wildlife reserve before lunch.","Grasslands stretched toward distant rocky hills."],
 ["A librarian replaced several damaged catalog cards.","Subway timetables changed after maintenance."]]),
("tiger","lion",
 ["an orange Asian wild cat marked by black stripes","a solitary jungle predator with dark body bands","a large striped feline in the forest"],
 ["a tawny big cat with a prominent mane","a social wild feline that lives in a pride","a large African carnivore famous for its mane"],
 [
 ["Orange fur with black stripes moved between the trees.","A solitary striped big cat stalked prey in dense forest."],
 ["A powerful wild feline prowled through an Asian jungle.","Zookeepers fed a large carnivorous cat."],
 ["The rainforest path was damp after a storm.","A wildlife photographer adjusted a forest camera."],
 ["A passenger checked the terminal departure board.","A contractor installed pipes beneath a road."]]),
("piano","violin",
 ["a keyboard instrument with black and white keys","an acoustic instrument whose strings are struck by hammers","a large instrument played using both hands on keys"],
 ["a bowed orchestral instrument held near the shoulder","a small four-string instrument played with a bow","a high pitched classical string instrument"],
 [
 ["A musician pressed black and white keys across the keyboard.","A grand acoustic keyboard instrument stood on stage."],
 ["A student practiced chords in the music room.","The hall hosted a classical solo performance."],
 ["The audience entered the theater before the lights dimmed.","A composer examined pages of printed notation."],
 ["A mechanic repaired the engine of a bus.","Rainwater collected behind the garage."]]),
("coffee","tea",
 ["a dark beverage made from ground roasted beans","an espresso-based caffeinated drink","a bitter aromatic brew served in cafes"],
 ["an infusion made by steeping dried leaves","a leaf-based hot drink","a drink brewed from plant leaves rather than roasted beans"],
 [
 ["Roasted beans were ground before hot water was poured.","A bitter dark drink steamed from an espresso cup."],
 ["A barista prepared an order in the morning cafe.","Freshly brewed drinks filled a small shop with aroma."],
 ["Two friends sat at a table downtown.","A commuter stopped for breakfast on the way to work."],
 ["An engineer inspected a bridge's steel supports.","Children sorted pebbles by a dry creek."]]),
("ocean","river",
 ["a vast saltwater body spanning continents","an immense stretch of open seawater","a saline expanse where large ships sail"],
 ["a flowing freshwater channel through land","a watercourse moving between banks","a long inland current that runs toward the sea"],
 [
 ["Saltwater waves rolled toward an endless horizon.","A vast expanse of seawater separated continents."],
 ["A ship sailed beyond the coastline toward open water.","A sailor studied tides before departing from port."],
 ["A coastal village prepared for tourist season.","An old lighthouse stood on a rocky cliff."],
 ["A teacher rearranged desks before class.","A bakery sold fresh bread at dawn."]]),
("castle","palace",
 ["a defensive medieval stone fortress","a stronghold with battlements and thick walls","a fortified old building with a moat and towers"],
 ["an ornate royal residence with grand halls","an opulent residence used by a monarch","a luxurious court building for official ceremonies"],
 [
 ["A medieval fortress had thick walls and tall watchtowers.","Battlements and a deep moat surrounded the stronghold."],
 ["Knights returned to a fortified royal residence.","Visitors toured an old stone keep guarded by soldiers."],
 ["A medieval village held a street festival.","An ancient hillside settlement welcomed tourists."],
 ["A dentist checked the appointment register.","A gardener planted seedlings in spring."]]),
("bicycle","scooter",
 ["a pedal-powered two-wheeled road vehicle","a human powered machine with a saddle and chain","a wheeled vehicle propelled by turning foot pedals"],
 ["a small footboard vehicle steered by handlebars","a compact kick-powered personal vehicle","a two wheeled standing device with a narrow platform"],
 [
 ["A rider pedaled a two-wheeled vehicle along the lane.","A metal frame with pedals and handlebars leaned against a wall."],
 ["A commuter wore a helmet on the morning ride.","The repair shop adjusted a chain and brake cables."],
 ["A city built lanes beside a busy street.","A fitness club hosted an outdoor endurance event."],
 ["A chef simmered vegetables in a soup pot.","A violinist rehearsed a short piece backstage."]])
]
assert len(ITEMS)==8
def has_word(s,w):
    return bool(re.search(r"(?<![A-Za-z])"+re.escape(w)+r"(?![A-Za-z])",s,re.I))
cue_rows=[]
for x,y,xd,yd,level_sets in ITEMS:
    assert len(xd)==len(yd)==3 and len(level_sets)==4
    for level,sentences in zip(LEVELS,level_sets):
        assert len(sentences)==2
        for cue_id,s in enumerate(sentences,1):
            assert not has_word(s,x) and not has_word(s,y),(x,s)
            cue_rows.append(dict(concept=x,replacement=y,level=level,
                                 cue_id=cue_id,cue_text=s))
cues=pd.DataFrame(cue_rows)
assert len(cues)==64
display(cues.head(6))

## 2 — Independent semantic distance (MiniLM)

Each cue's distance to X is estimated as one minus the cosine similarity between a separate MiniLM encoder's embeddings of a target phrase and the cue sentence. **Proposed near/medium/far/unrelated groups are not factual ground truth.** Order failures are logged. The continuous measured distance is the primary predictor; human ratings should be collected before making stronger claims.

In [ ]:
embed_tok=AutoTokenizer.from_pretrained(EMBED_ID,trust_remote_code=False)
embed_model=AutoModel.from_pretrained(EMBED_ID,trust_remote_code=False).cpu().eval()
concepts=[item[0] for item in ITEMS]
texts=[f"This text concerns {x}." for x in concepts]+cues.cue_text.tolist()
vecs=[]
for start in tqdm(range(0,len(texts),32),desc="MiniLM embeddings"):
    batch=embed_tok(texts[start:start+32],return_tensors="pt",padding=True,truncation=True,max_length=128)
    with torch.inference_mode():
        h=embed_model(**batch).last_hidden_state
        mask=batch["attention_mask"].unsqueeze(-1)
        p=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
        vecs.append(F.normalize(p.float(),dim=-1).cpu().numpy())
vecs=np.concatenate(vecs,axis=0)
reference={concept:vecs[i] for i,concept in enumerate(concepts)}
cues["cosine_similarity"]=[float(reference[row.concept]@vecs[len(concepts)+idx])
                           for idx,row in cues.reset_index(drop=True).iterrows()]
cues["semantic_distance"]=1-cues.cosine_similarity
cues["word_count"]=cues.cue_text.str.split().str.len()
save_csv(cues,"01_cue_distances.csv")
level_distance=cues.groupby(["concept","level"],as_index=False).semantic_distance.mean()
order=level_distance.pivot(index="concept",columns="level",values="semantic_distance").reindex(columns=LEVELS)
order["ordered_as_proposed"]=np.all(np.diff(order[LEVELS].values,axis=1)>0,axis=1)
save_csv(order.reset_index(),"02_distance_order_diagnostics.csv")
display(order.round(3))
print("Distance levels ordered for",int(order.ordered_as_proposed.sum()),"/",len(order),"concepts.")
if not order.ordered_as_proposed.all():
    print("WARNING: Proposed groups are not consistently distance ordered. Continuous measured distances take precedence.")
del embed_model,embed_tok,vecs
if torch.cuda.is_available():torch.cuda.empty_cache()

## 3 — Load Qwen and match the hidden-state readout position

The hidden vector is taken at the **last token of the assistant-generation header** added by Qwen's chat template. That is the same structural token across probe training and all test prompts. Although we append a common `Topic:` marker in the user text, the actual measured token comes *after* that marker. Variable input length and differences between instruction wording remain potential confounders.

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,dtype=DTYPE,trust_remote_code=False,low_cpu_mem_usage=True).to(DEVICE).eval()
assert model.config.num_hidden_layers==24
assert max(LAYERS)<=model.config.num_hidden_layers
def chat(s):
    return tok.apply_chat_template([{"role":"user","content":s}],
                                   tokenize=False,add_generation_prompt=True)
@torch.inference_mode()
def features(prompt):
    enc=tok(chat(prompt),return_tensors="pt",add_special_tokens=False).to(DEVICE)
    hs=model(**enc,output_hidden_states=True,return_dict=True,use_cache=False).hidden_states
    return {layer:hs[layer][0,-1,:].float().cpu().numpy() for layer in LAYERS},int(enc.input_ids.shape[1])
testfeat,testlen=features("The topic word is banana.\nTopic:")
assert len(testfeat)==len(LAYERS) and all(np.isfinite(v).all() for v in testfeat.values())
print("Model ready:",MODEL_ID,"input tokens:",testlen,"hidden dimension:",testfeat[24].size)

## 4 — Train independent X-vs-Y probes and validate on unseen descriptions

Train 8 named templates × 2 classes = 16 examples per concept. Validate separately on 3 novel named templates × 2 classes, and on 3 **different name-free descriptions per class** × 2 held-out frames = 12 evaluation contexts. Descriptions are related paraphrases, not 12 statistically independent semantic concepts. No test cues are used in probe fitting. AUC ≥ 0.75 is the prespecified exploratory screen, and failing probes' data are **still saved**.

In [ ]:
TRAIN_FRAMES=[
"The note concerns {}.\nTopic:","Think of {}.\nTopic:",
"An example topic is {}.\nTopic:","Someone mentioned {} earlier.\nTopic:",
"The main subject is {}.\nTopic:","The concept to discuss is {}.\nTopic:",
"Keep {} in mind.\nTopic:","Focus attention on {}.\nTopic:"]
NAMED_TEST_FRAMES=[
"A writer spoke about {}.\nTopic:","The conversation revolves around {}.\nTopic:",
"The word of interest is {}.\nTopic:"]
DESC_TEST_FRAMES=["An exhibit concerns {}.\nTopic:","A document describes {}.\nTopic:"]
probe_inputs=[]
for x,y,x_desc,y_desc,_ in tqdm(ITEMS,desc="Probe fit and holdout inputs"):
    for label,word,descriptions in [(1,x,x_desc),(0,y,y_desc)]:
        for f in TRAIN_FRAMES:
            p=f.format(word);h,n=features(p)
            probe_inputs.append(dict(concept=x,split="train",label=label,prompt=p,features=h))
        for f in NAMED_TEST_FRAMES:
            p=f.format(word);h,n=features(p)
            probe_inputs.append(dict(concept=x,split="named_holdout",label=label,prompt=p,features=h))
        for desc in descriptions:
            assert not has_word(desc,x) and not has_word(desc,y),(x,desc)
            for f in DESC_TEST_FRAMES:
                p=f.format(desc);h,n=features(p)
                probe_inputs.append(dict(concept=x,split="description_holdout",label=label,prompt=p,features=h))
probe_inputs=pd.DataFrame(probe_inputs)
assert probe_inputs.groupby("split").size().to_dict()=={
    "train":128,"named_holdout":48,"description_holdout":96}
probes={};metrics=[]
for x in concepts:
    ex=probe_inputs[probe_inputs.concept==x]
    train=ex[ex.split=="train"]
    probes[x]={}
    for layer in LAYERS:
        X=np.stack([a[layer] for a in train.features]);y=train.label.to_numpy()
        clf=make_pipeline(StandardScaler(),LogisticRegression(
            C=0.1,solver="liblinear",random_state=SEED,max_iter=2000))
        clf.fit(X,y)
        sd=max(float(clf.decision_function(X).std(ddof=1)),1e-8)
        probes[x][layer]=(clf,sd)
        for split in ["named_holdout","description_holdout"]:
            ev=ex[ex.split==split];V=np.stack([a[layer] for a in ev.features])
            score=clf.decision_function(V)
            metrics.append(dict(concept=x,layer=layer,split=split,n=len(ev),
                auc=float(roc_auc_score(ev.label.to_numpy(),score)),
                accuracy=float(((score>=0).astype(int)==ev.label.to_numpy()).mean()),
                train_decision_sd=sd))
validation=pd.DataFrame(metrics)
valid=validation[validation.split=="description_holdout"].copy()
valid["validated"]=valid.auc>=MIN_DESCRIPTION_AUC
save_csv(validation,"03_probe_validation.csv")
save_csv(valid,"04_probe_validated_flags.csv")
display(valid.groupby("layer").agg(mean_auc=("auc","mean"),passed=("validated","sum"),total=("validated","size")).round(3))
del probe_inputs
print("Do not interpret scores from layers with poor held-out validation as semantic concept activation.")